# Chapter 58: Multimedia Web Applications & APIs
## Multimedia & Computer Vision Learning Hub — ITERA
**Author:** Aji Pamungkas Tri Nurcahyo  
**Objective:** Menganalisis arsitektur pemrosesan multimedia berbasis peramban (*in-browser multimedia processing*), menghitung jejak memori frame buffer RGBA dan throughput bandwidth memori, mengevaluasi trade-off resolusi spasial terhadap laju frame, serta memvalidasi efisiensi pemisahan thread komputasi menggunakan Web Workers.

### 1. Landasan Matematis: Jejak Memori Buffer & Bandwidth Bus Peramban
Ukuran Byte per Frame (RGBA Interleaved):
$$\text{Bytes} = W \times H \times 4$$

Ukuran Memori (Megabyte biner):
$$\text{Footprint (MB)} = \frac{W \times H \times 4}{1024 \times 1024}$$

Throughput Piksel (Megapiksel per detik):
$$\text{Throughput} = \frac{W \times H \times F}{10^6}$$

Kebutuhan Bandwidth Memori (MB per detik):
$$\text{Bandwidth} = \text{Footprint (MB)} \times F$$

In [ ]:
# 2. Setup Lingkungan & Dual-Engine Check
import numpy as np
import sys

print(f"Python Version: {sys.version}")
print(f"NumPy Version: {np.__version__}")

HAS_TORCH = False
try:
    import torch
    HAS_TORCH = True
    print(f"PyTorch: {torch.__version__} (Device: {'cuda' if torch.cuda.is_available() else 'cpu'})")
except ImportError:
    print("INFO: PyTorch tidak terpasang di host lokal. Beroperasi dalam Mode Analisis Rekayasa Web Multimedia NumPy murni.")

In [ ]:
# 3. Implementasi Metrik Buffer Memori & Validasi Seksi 3
def calc_buffer_metrics(w, h, fps=30.0, c=4):
    n_bytes = w * h * c
    mb = n_bytes / (1024.0 * 1024.0)
    mp_sec = (w * h * fps) / 1e6
    bw_mb_sec = mb * fps
    return n_bytes, mb, mp_sec, bw_mb_sec

# Skenario A: 1080p pada 60 FPS
bytes_fhd, mb_fhd, mp_fhd, bw_fhd = calc_buffer_metrics(1920, 1080, fps=60.0)
print(f"1080p 60 FPS: {bytes_fhd:,} Bytes | {mb_fhd:.3f} MB/frame | {mp_fhd:.2f} MP/s | {bw_fhd:.2f} MB/s")
assert bytes_fhd == 8_294_400
np.testing.assert_allclose(mb_fhd, 7.910156, atol=1e-4)
np.testing.assert_allclose(mp_fhd, 124.416, atol=1e-3)
np.testing.assert_allclose(bw_fhd, 474.609, atol=1e-2)

# Skenario B: VGA 640x480 pada 30 FPS
bytes_vga, mb_vga, mp_vga, bw_vga = calc_buffer_metrics(640, 480, fps=30.0)
print(f"VGA 30 FPS: {bytes_vga:,} Bytes | {mb_vga:.3f} MB/frame | {bw_vga:.2f} MB/s")
assert bytes_vga == 1_228_800
np.testing.assert_allclose(mb_vga, 1.171875, atol=1e-4)
np.testing.assert_allclose(bw_vga, 35.15625, atol=1e-3)

savings = (1.0 - bw_vga / bw_fhd) * 100.0
print(f"Penghematan Bandwidth VGA vs 1080p: {savings:.2f}%")
assert savings > 90.0
print("VALIDASI METRIK BUFFER & BANDWIDTH: Lulus sempurna!")

In [ ]:
# 4. Komparasi Spektrum Resolusi Peramban (4K, 1080p, 720p, VGA, QVGA)
resolutions = [
    ("4K UHD (2160p)", 3840, 2160, 60.0),
    ("Full HD (1080p)", 1920, 1080, 60.0),
    ("HD (720p)", 1280, 720, 30.0),
    ("Standard VGA", 640, 480, 30.0),
    ("Mobile QVGA", 320, 240, 30.0)
]

print(f"{'Resolusi Format':<18} | {'Dimensi':<10} | {'FPS':<4} | {'MB/Frame':<8} | {'Bandwidth (MB/s)':<16}")
print("-" * 68)
for name, w, h, fps in resolutions:
    _, mb, _, bw = calc_buffer_metrics(w, h, fps=fps)
    print(f"{name:<18} | {w}x{h:<6} | {fps:<4.0f} | {mb:<8.2f} | {bw:<16.2f}")
print("VERIFIKASI TABEL SPEKTRUM RESOLUSI: Selesai.")

In [ ]:
# 5. Simulasi Pemisahan Thread: Main Thread vs Web Worker
def simulate_thread_blocking(t_compute=25.0, t_transfer=1.5, budget=16.67):
    main_blocking = t_compute
    worker_blocking = t_transfer
    main_ok = main_blocking <= budget
    worker_ok = worker_blocking <= budget
    reduction = ((main_blocking - worker_blocking) / main_blocking) * 100.0
    return main_blocking, main_ok, worker_blocking, worker_ok, reduction

mb, mok, wb, wok, red = simulate_thread_blocking(25.0, 1.5)
print(f"Main Thread Blocking: {mb:.1f} ms | Bebas Stutter: {mok}")
print(f"Web Worker Blocking: {wb:.1f} ms | Bebas Stutter: {wok}")
print(f"Reduksi Pemblokiran UI: {red:.2f}%")

assert mok is False, "25ms pada main thread membekukan frame 60 FPS"
assert wok is True, "1.5ms transfer overhead menjamin UI responsif 60 FPS"
assert red > 90.0
print("SIMULASI WEB WORKER: Berhasil membuktikan isolasi komputasi!")

### 6. Rangkuman & Pembahasan Akademik
1. **Beban Memori Real-Time**: Aliran video 1080p menuntut bandwidth memori hampir $0.5$ GB/s. Penggunaan resolusi inferensi proporsional (VGA) mereduksi beban bus hingga $>90%$.
2. **Isolasi Thread UI**: Web Worker mengeliminasi pembekuan antarmuka dengan memindahkan komputasi visi intensif ke thread latar belakang secara zero-copy.
3. **In-Browser Privacy**: Pemrosesan citra langsung di peramban klien menjamin privasi visual pengguna secara penuh tanpa biaya infrastruktur cloud.